# L5 Appendix. Send a copy to the cloud (optional)

Everything in L5 runs with no network. This notebook is the one path memory takes off the device, and it is off by default. Run L5 first so `assistant_shard` exists.


In [1]:
import os
from qdrant_edge import EdgeShard, ScrollRequest

# Uploading is a choice you make, never a default. To send a copy to
# your own Qdrant Cloud cluster: set UPLOAD_TO_CLOUD to True and put
# the cluster's URL and API key in the QDRANT_URL and QDRANT_API_KEY
# environment variables.
UPLOAD_TO_CLOUD = False

if not (UPLOAD_TO_CLOUD and os.getenv("QDRANT_URL")
        and os.getenv("QDRANT_API_KEY")):
    print("Nothing uploaded. Every memory stays on this device.")
else:
    from qdrant_client import QdrantClient, models

    client = QdrantClient(url=os.environ["QDRANT_URL"],
                          api_key=os.environ["QDRANT_API_KEY"])
    if client.collection_exists("assistant_memory"):
        print("assistant_memory already exists on the cluster.",
              "Delete it there first, or rename the collection here.")
    else:
        assistant_shard = EdgeShard.load("./assistant_shard")
        records, _ = assistant_shard.scroll(
            ScrollRequest(limit=1000,
                          with_payload=True,
                          with_vector=True)
        )
        client.create_collection(
            "assistant_memory",
            vectors_config={
                "text": models.VectorParams(
                    size=768, distance=models.Distance.COSINE),
                "image": models.VectorParams(
                    size=512, distance=models.Distance.COSINE),
            },
        )
        client.upsert("assistant_memory", points=[
            models.PointStruct(id=r.id, vector=r.vector,
                               payload=r.payload)
            for r in records
        ])
        print(f"Uploaded {len(records)} memories: same points,",
              "same format, readable by any Qdrant server",
              "or another device.")

Nothing uploaded. Every memory stays on this device.
